# 12 — U-Net source-plane reconstruction

Train a U-Net on `(observed_lensed_image, source_truth)` pairs. Once trained, the network undoes the lensing distortion in a single forward pass — a fully data-driven alternative to the classical pixelated source reconstruction.

In [ ]:
# Bootstrap: make `lensing` importable when running notebooks/ directly.
import sys
from pathlib import Path
repo = Path.cwd().resolve().parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

import lensing as gl
# Device-agnostic: prefer MPS (Apple GPU) → CUDA → CPU.
# Pass device="cpu" if you need to force the CPU path (e.g. for
# operators that have no MPS kernel yet, or for reproducibility).
device, dtype = gl.config.setup(seed=42)
print(f"using device: {device}")


In [ ]:
from torch.utils.data import DataLoader

train = gl.ml.datasets.LensSourcePairDataset(n_samples=500, npix=48,
                                              deltapix=0.05, seed=0)
val   = gl.ml.datasets.LensSourcePairDataset(n_samples=80, npix=48,
                                              deltapix=0.05, seed=10000)
train_loader = DataLoader(train, batch_size=16, shuffle=True)
val_loader   = DataLoader(val,   batch_size=16)

# Show a few examples
obs, src = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for j in range(4):
    gl.viz.imshow_log(obs[j, 0]+obs[j,0].min().abs()+1e-3,
                      ax=axes[0, j], title='observed')
    gl.viz.imshow_log(src[j, 0]+src[j,0].min().abs()+1e-3,
                      ax=axes[1, j], title='true source')
plt.tight_layout(); plt.show()


## Train the U-Net

In [ ]:
model = gl.ml.models.UNet(in_channels=1, out_channels=1, base=16)
print(f'{sum(p.numel() for p in model.parameters()):,} parameters')
history = gl.ml.train.fit_model(
    model, train_loader, val_loader,
    loss_fn=nn.MSELoss(),
    lr=1e-3, epochs=8,
    log_every=1,
)


## Test reconstructions

In [ ]:
test = gl.ml.datasets.LensSourcePairDataset(n_samples=8, npix=48,
                                             deltapix=0.05, seed=999)
obs = torch.stack([test[i][0] for i in range(8)])
src = torch.stack([test[i][1] for i in range(8)])
model.eval()
with torch.no_grad():
    pred = model(obs.to(device)).cpu()

fig, axes = plt.subplots(3, 8, figsize=(20, 8))
for j in range(8):
    gl.viz.imshow_log(obs[j, 0]+obs[j,0].min().abs()+1e-3,
                      ax=axes[0, j], title='observed' if j==0 else None)
    gl.viz.imshow_log(pred[j, 0]+pred[j,0].min().abs()+1e-3,
                      ax=axes[1, j], title='U-Net' if j==0 else None)
    gl.viz.imshow_log(src[j, 0]+src[j,0].min().abs()+1e-3,
                      ax=axes[2, j], title='truth' if j==0 else None)
    for ax in axes[:, j]:
        ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()


## Image-quality validation

Two standard image-regression metrics, computed per test
image and reported as mean ± std on a 32-image set:

* **PSNR** (Peak Signal-to-Noise Ratio, dB):
  ``20·log₁₀(data_range / RMSE)``. > 30 dB is
  visually indistinguishable for natural images;
  for U-Net source reconstruction we expect 25–35 dB.
* **SSIM** (Wang+ 2004): structural similarity index
  ∈ [-1, 1], 1 = identical. SSIM ≳ 0.9 means the
  reconstruction preserves the source morphology well.

In [ ]:
test_set = gl.ml.datasets.LensSourcePairDataset(n_samples=32, npix=48,
                                                 deltapix=0.05, seed=42)
obs_t = torch.stack([test_set[i][0] for i in range(32)])
src_t = torch.stack([test_set[i][1] for i in range(32)])
model.eval()
with torch.no_grad():
    pred_t = model(obs_t.to(device)).cpu()

fig, _, summary = gl.viz.diagnostics.plot_image_quality(
    src_t.numpy(), pred_t.numpy(), n_show=6,
    title='U-Net source reconstruction (truth, pred, |diff|)',
)
plt.show()
print(gl.viz.diagnostics.format_summary({
    'PSNR mean [dB]': summary['psnr_mean'],
    'PSNR std  [dB]': summary['psnr_std'],
    'SSIM mean'     : summary['ssim_mean'],
    'SSIM std'      : summary['ssim_std'],
    'n test images' : len(obs_t),
}, title='U-Net validation metrics'))
